# 🎵 AISO MusicGen — Colab API Server v1.0

Bu notebook’u Google Colab’da **GPU runtime** ile çalıştırın.

1. **Runtime > Change runtime type > T4 GPU** seçin
2. Tüm hücreleri sırayla çalıştırın
3. Son hücredeki **public URL**’yi kopyalayın
4. AISO Unified uygulamasına yapıştırın

### Desteklenen Modeller
| Model | VRAM | Hız | Özellik |
|-------|------|------|----------|
| `musicgen-small` | ~1 GB | ⭐⭐⭐ | Hızlı, temel kalite |
| `musicgen-medium` | ~3.3 GB | ⭐⭐ | Dengeli, iyi kalite |
| `musicgen-melody` | ~3.3 GB | ⭐⭐ | Melodi koşullu üretim |

In [ ]:
# ══ 1. Bağımlılıkları Kur ═══════════════════════════════════════
!pip install -q fastapi uvicorn transformers scipy accelerate
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
print("✅ Bağımlılıklar kuruldu")

In [ ]:
# ══ 2. Yapılandırma + Model Yükle ═══════════════════════════════
# ▼▼▼ Modeli değiştirmek için bu satırı düzenleyin ▼▼▼
MODEL_ID = "facebook/musicgen-small"  # small | medium | melody

import torch, io, time, re, json, subprocess, threading
import numpy as np
import scipy.io.wavfile
from transformers import (
    AutoProcessor,
    MusicgenForConditionalGeneration,
    MusicgenMelodyForConditionalGeneration
)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"🖥 Device: {device} | GPU: {torch.cuda.get_device_name(0)} | VRAM: {vram:.1f} GB")
else:
    print(f"🖥 Device: {device} (GPU yok — yavaş olacak)")

print(f"⏳ Model yükleniyor: {MODEL_ID}...")
if "melody" in MODEL_ID:
    model = MusicgenMelodyForConditionalGeneration.from_pretrained(MODEL_ID).to(device)
else:
    model = MusicgenForConditionalGeneration.from_pretrained(MODEL_ID).to(device)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print(f"✅ Model yüklendi: {MODEL_ID} on {device}")

In [ ]:
# ══ 3. FastAPI Server Tanımla ═══════════════════════════════════
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import Response, JSONResponse
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="AISO MusicGen API", version="1.0")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], allow_methods=["*"], allow_headers=["*"]
)

def _generate_audio(prompt, duration=15, guidance_scale=3.0,
                    temperature=1.0, top_k=250, melody_waveform=None, melody_sr=None):
    """Ortak üretim fonksiyonu"""
    max_tokens = min(duration * 50, 1500)  # 50 token/sn, maks 30sn

    # Melody-conditioned mı?
    if melody_waveform is not None and "melody" in MODEL_ID:
        inputs = processor(
            text=[prompt],
            audio=melody_waveform,
            sampling_rate=melody_sr,
            padding=True, return_tensors="pt"
        ).to(device)
    else:
        inputs = processor(text=[prompt], padding=True, return_tensors="pt").to(device)

    gen_kwargs = dict(max_new_tokens=max_tokens, guidance_scale=guidance_scale)
    if temperature > 0:
        gen_kwargs.update(do_sample=True, temperature=temperature, top_k=top_k)

    with torch.no_grad():
        audio_values = model.generate(**inputs, **gen_kwargs)

    samples = audio_values[0, 0].cpu().numpy()
    sr = model.config.audio_encoder.sampling_rate

    # WAV bytes
    buf = io.BytesIO()
    wav_data = (np.clip(samples, -1, 1) * 32767).astype(np.int16)
    scipy.io.wavfile.write(buf, sr, wav_data)
    buf.seek(0)

    torch.cuda.empty_cache()
    return buf.read(), sr, len(samples) / sr

# ── Endpoints ────────────────────────────────────────
@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": MODEL_ID,
        "device": str(device),
        "is_melody": "melody" in MODEL_ID
    }

@app.post("/generate")
async def generate(
    prompt: str = Form(...),
    duration: int = Form(15),
    guidance_scale: float = Form(3.0),
    temperature: float = Form(1.0),
    top_k: int = Form(250)
):
    try:
        wav_bytes, sr, dur = _generate_audio(
            prompt, duration, guidance_scale, temperature, top_k
        )
        return Response(
            content=wav_bytes,
            media_type="audio/wav",
            headers={"X-Sample-Rate": str(sr), "X-Duration": f"{dur:.1f}"}
        )
    except Exception as e:
        torch.cuda.empty_cache()
        raise HTTPException(500, detail=str(e))

@app.post("/generate_melody")
async def generate_melody(
    prompt: str = Form(...),
    duration: int = Form(15),
    guidance_scale: float = Form(3.0),
    audio_file: UploadFile = File(...)
):
    if "melody" not in MODEL_ID:
        raise HTTPException(400, "Melody modeli yüklü değil. Hücre 2'de MODEL_ID='facebook/musicgen-melody' yapın.")
    try:
        import torchaudio
        audio_bytes = await audio_file.read()
        waveform, sr_orig = torchaudio.load(io.BytesIO(audio_bytes))
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        wav_bytes, sr, dur = _generate_audio(
            prompt, duration, guidance_scale,
            melody_waveform=waveform.squeeze(0).numpy(),
            melody_sr=sr_orig
        )
        return Response(
            content=wav_bytes,
            media_type="audio/wav",
            headers={"X-Sample-Rate": str(sr), "X-Duration": f"{dur:.1f}"}
        )
    except Exception as e:
        torch.cuda.empty_cache()
        raise HTTPException(500, detail=str(e))

@app.post("/switch_model")
async def switch_model(model_id: str = Form(...)):
    global model, processor, MODEL_ID
    valid = ["facebook/musicgen-small", "facebook/musicgen-medium", "facebook/musicgen-melody"]
    if model_id not in valid:
        raise HTTPException(400, f"Geçersiz model. Seçenekler: {valid}")
    try:
        del model
        torch.cuda.empty_cache()
        MODEL_ID = model_id
        if "melody" in model_id:
            model = MusicgenMelodyForConditionalGeneration.from_pretrained(model_id).to(device)
        else:
            model = MusicgenForConditionalGeneration.from_pretrained(model_id).to(device)
        processor = AutoProcessor.from_pretrained(model_id)
        return {"status": "ok", "model": model_id}
    except Exception as e:
        torch.cuda.empty_cache()
        raise HTTPException(500, detail=str(e))

print("✅ FastAPI endpoints tanımlandı: /health, /generate, /generate_melody, /switch_model")

In [ ]:
# ══ 4. Server + Tunnel Başlat ═══════════════════════════════════
import uvicorn

# 1) Uvicorn’u arka planda başlat
server_thread = threading.Thread(
    target=uvicorn.run,
    args=(app,),
    kwargs={"host": "0.0.0.0", "port": 8000, "log_level": "warning"},
    daemon=True
)
server_thread.start()
time.sleep(2)
print("✅ FastAPI server çalışıyor (port 8000)")

# 2) Cloudflared tünel başlat (hesap gerektirmez!)
print("⏳ Cloudflare tünel açılıyor...")
tunnel_proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

# 3) Tünel URL’sini oku
tunnel_url = None
for _ in range(30):
    line = tunnel_proc.stderr.readline().decode()
    if not line:
        time.sleep(1)
        continue
    m = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
    if m:
        tunnel_url = m.group(1)
        break

if tunnel_url:
    print()
    print('=' * 64)
    print(f'  🌐 AISO API URL: {tunnel_url}')
    print('=' * 64)
    print()
    print('Bu URL\'yi AISO Unified uygulamasına yapıştırın!')
    print(f'Test: {tunnel_url}/health')
    print()
    print('🟢 Server çalışıyor. Bu hücreyi DURDURMAY IN!')
    print('Kapatmak için: Runtime > Disconnect and delete runtime')
else:
    print('⚠️ Tünel URL alınamadı. Hücreyi tekrar çalıştırın.')

# Server’ı canlı tut
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print('\n⏹ Server kapatıldı.')